In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Sample text data
text = """
It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity, it was the season of Light, it was the season of Darkness, it was the spring of hope, it was the winter of despair, we had everything before us, we had nothing before us, we were all going direct to Heaven, we were all going direct the other way – in short, the period was so far like the present period, that some of its noisiest authorities insisted on its being received, for good or for evil, in the superlative degree of comparison only.
"""

text += """
The project of bringing an oceanic trunk line, with an adequate number of service branches, to every important industrial and commercial center in the United States, with an eye to every stage of development in the history of a product, was the last of the most important great enterprises to be undertaken by a company that had already taken a very important part in the development of the telegraph and telephone systems.
"""

text += """
The sun shone, having no alternative, on the nothing new in this world. There was something so idyllic in the surroundings that it was impossible for anyone to remain depressed in this place. The meadow, the stream, the gentle breeze, and the distant hills created an ambiance that seemed to soothe even the most troubled soul.
"""

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []
for i in range(1, len(text.split())):
    sequence = tokenizer.texts_to_sequences([text])[0]
    for j in range(1, len(sequence)):
        n_gram_sequence = sequence[:j+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_length = max(len(x) for x in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre')
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)


In [3]:
model = Sequential()
model.add(Embedding(total_words, 50, input_length=X.shape[1]))
model.add(LSTM(100, return_sequences=True))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


C:\Users\NEHA\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [4]:
model.fit(X, y, epochs=2, verbose=1)

Epoch 1/2
1907/1907 ━━━━━━━━━━━━━━━━━━━━ 761s 395ms/step - accuracy: 0.6833 - loss: 1.4627 
Epoch 2/2
1907/1907 ━━━━━━━━━━━━━━━━━━━━ 516s 270ms/step - accuracy: 0.9966 - loss: 0.0586


In [5]:
def generate_text(seed_text, next_words, model, tokenizer, max_sequence_length):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_length-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted_probs, axis=-1)[0]
        predicted_word = tokenizer.index_word[predicted_word_index]
        seed_text += " " + predicted_word
    return seed_text

# Generate new text
seed_text = "It was the best"
generated_text = generate_text(seed_text, 5, model, tokenizer, max_sequence_length)
print(generated_text)


It was the best of times it was the
